In [20]:
import os
import numpy as np
import pandas as pd
import datasets
from scipy import stats
from tqdm.auto import tqdm

DATA_DIR = "../data/hf_dataset"


In [21]:
def per_site_stats(target: list) -> dict:
    """Compute stats for a single site's time series."""
    v = np.array(target, dtype=np.float32)
    v = v[~np.isnan(v)]
    if len(v) == 0:
        return {k: np.nan for k in ["mean", "median", "std", "skewness", "p10", "p90"]}
    return {
        "mean":     float(np.mean(v)),
        "median":   float(np.median(v)),
        "std":      float(np.std(v)),
        "skewness": float(stats.skew(v)),
        "p10":      float(np.percentile(v, 10)),
        "p90":      float(np.percentile(v, 90)),
    }


In [22]:
networks = [n for n in sorted(os.listdir(DATA_DIR)) if n != "CNEMC"]
all_rows = []

for network in tqdm(networks, desc="Networks"):
    ds_path = os.path.join(DATA_DIR, network, "H")
    ds_df = datasets.load_from_disk(ds_path).to_pandas()

    # rsplit so site codes with underscores don't bleed into the pollutant name
    ds_df["pollutant"] = ds_df["item_id"].str.rsplit("_", n=1).str[-1]

    # Compute stats per site, then mean-aggregate over sites per pollutant
    site_stats = pd.DataFrame(
        [per_site_stats(row.target) for row in ds_df.itertuples()],
        index=ds_df.index,
    )
    site_stats["pollutant"] = ds_df["pollutant"].values

    agg = site_stats.groupby("pollutant").mean()
    agg.insert(0, "network", network)
    all_rows.append(agg.reset_index())

results_df = pd.concat(all_rows, ignore_index=True).set_index(["network", "pollutant"])
results_df


Networks: 100%|██████████| 7/7 [00:07<00:00,  1.12s/it]


mean      median        std   skewness  \
network     pollutant                                                 
AURN        CO           0.190580    0.151346   0.141710   4.557207   
            NO2         17.564643   14.265011  12.954172   1.736060   
            Ozone       52.474240   53.712868  22.093827   0.121841   
            PM10        13.409365   11.171437   9.586900   4.151592   
            PM2.5        7.453621    5.640341   6.294068   3.258931   
            SO2          1.117915    0.575638   3.538700  23.051280   
CNEMC_SMALL CO           0.630936    0.570390   0.295637   1.856816   
            NO2         22.524634   17.984788  16.098228   1.639518   
            Ozone       67.535797   62.174720  40.786257   0.702164   
            PM10        62.159950   47.502481  65.683413   8.013429   
            PM2.5       31.090281   23.431334  28.944389   4.800305   
            SO2          8.279264    7.020385   5.708826   7.941761   
CPCB        CO           0.860024    0.703634   0.659080   2.898113   
            NO2         24.656347   19.451824  19.777036   3.413465   
            Ozone       29.180047   21.711454  25.147445   2.210509   
            PM10       122.114443  102.840171  84.885456   2.500329   
            PM2.5       57.070293   42.509327  50.156842   3.864121   
            SO2         13.429687   11.307478   9.818385   4.254236   
EEA_DE      CO           0.280303    0.250282   0.134522   2.633840   
            NO2         15.074937   12.774760   9.760157   1.676931   
            Ozone       53.456144   52.756103  27.789786   0.352647   
            PM10        14.217957   11.933893  10.547070   6.465004   
            PM2.5        8.922321    6.946073   7.546064   5.380876   
            SO2          1.854043    1.282205   2.574617  10.855451   
EEA_FR      CO           0.224498    0.198400   0.129853   2.524663   
            NO2         14.238802   11.429780  10.538208   2.013134   
            Ozone       56.434490   56.722733  26.340310   0.218415   
            PM10        15.016806   12.588745  10.589692   3.590955   
            PM2.5        8.625694    6.628234   7.305961   3.549489   
            SO2          2.244792    1.094467   5.680895  15.702466   
EPA         CO           0.325700    0.269960   0.202970   2.996143   
            NO2         15.472422   11.723837  12.242189   1.947218   
            Ozone       62.015795   61.754847  27.241017   0.210240   
            PM10        22.231302   16.506340  31.792301  14.117332   
            PM2.5        7.722436    6.166343   7.697349   9.192781   
            SO2          2.576870    1.243150   5.389112   9.879477   
SINAICA     CO           1.410447    1.108984   9.674176  23.104192   
            NO2         29.816515   24.114709  20.453552   1.648694   
            Ozone       52.472077   44.968885  42.752793   4.246500   
            PM10        61.196215   48.451813  54.164352   4.689562   
            PM2.5       22.504307   15.833456  34.270618   5.516422   
            SO2         12.200468   10.061141  11.723262   7.194940   

                             p10         p90  
network     pollutant                         
AURN        CO          0.093136    0.314334  
            NO2         4.775718   34.975940  
            Ozone      22.174245   78.229779  
            PM10        4.778231   24.505191  
            PM2.5       2.356798   14.771238  
            SO2         0.208508    1.717962  
CNEMC_SMALL CO          0.346264    0.989120  
            NO2         6.975466   44.711068  
            Ozone      18.234597  123.872071  
            PM10       18.672775  115.551570  
            PM2.5       8.224084   62.235570  
            SO2         3.952920   13.649084  
CPCB        CO          0.279750    1.589494  
            NO2         7.845825   47.623818  
            Ozone       6.929282   61.537905  
            PM10       38.470343  228.370780  
            PM2.5      14.814644  117.646948  
           

In [23]:
# Optional: save to CSV
results_df.to_csv("../output/data_statistics.csv")
print("Saved to ../output/data_statistics.csv")
results_df.round(3)


Saved to ../output/data_statistics.csv


mean   median     std  skewness     p10      p90
network     pollutant                                                     
AURN        CO           0.191    0.151   0.142     4.557   0.093    0.314
            NO2         17.565   14.265  12.954     1.736   4.776   34.976
            Ozone       52.474   53.713  22.094     0.122  22.174   78.230
            PM10        13.409   11.171   9.587     4.152   4.778   24.505
            PM2.5        7.454    5.640   6.294     3.259   2.357   14.771
            SO2          1.118    0.576   3.539    23.051   0.209    1.718
CNEMC_SMALL CO           0.631    0.570   0.296     1.857   0.346    0.989
            NO2         22.525   17.985  16.098     1.640   6.975   44.711
            Ozone       67.536   62.175  40.786     0.702  18.235  123.872
            PM10        62.160   47.502  65.683     8.013  18.673  115.552
            PM2.5       31.090   23.431  28.944     4.800   8.224   62.236
            SO2          8.279    7.020   5.709     7.942   3.953   13.649
CPCB        CO           0.860    0.704   0.659     2.898   0.280    1.589
            NO2         24.656   19.452  19.777     3.413   7.846   47.624
            Ozone       29.180   21.711  25.147     2.211   6.929   61.538
            PM10       122.114  102.840  84.885     2.500  38.470  228.371
            PM2.5       57.070   42.509  50.157     3.864  14.815  117.647
            SO2         13.430   11.307   9.818     4.254   5.202   23.340
EEA_DE      CO           0.280    0.250   0.135     2.634   0.157    0.435
            NO2         15.075   12.775   9.760     1.677   5.132   28.160
            Ozone       53.456   52.756  27.790     0.353  16.755   89.970
            PM10        14.218   11.934  10.547     6.465   5.065   25.480
            PM2.5        8.922    6.946   7.546     5.381   2.711   17.325
            SO2          1.854    1.282   2.575    10.855   0.681    3.170
EEA_FR      CO           0.224    0.198   0.130     2.525   0.099    0.375
            NO2         14.239   11.430  10.538     2.013   4.269   28.035
            Ozone       56.434   56.723  26.340     0.218  20.863   89.419
            PM10        15.017   12.589  10.590     3.591   5.483   27.055
            PM2.5        8.626    6.628   7.306     3.549   2.586   16.819
            SO2          2.245    1.094   5.681    15.702   0.206    4.365
EPA         CO           0.326    0.270   0.203     2.996   0.149    0.568
            NO2         15.472   11.724  12.242     1.947   3.965   32.741
            Ozone       62.016   61.755  27.241     0.210  25.864   97.419
            PM10        22.231   16.506  31.792    14.117   6.018   40.669
            PM2.5        7.722    6.166   7.697     9.193   2.279   14.191
            SO2          2.577    1.243   5.389     9.879   0.231    5.714
SINAICA     CO           1.410    1.109   9.674    23.104   0.499    2.200
            NO2         29.817   24.115  20.454     1.649  10.607   57.229
            Ozone       52.472   44.969  42.753     4.246  11.470  100.208
            PM10        61.196   48.452  54.164     4.690  23.039  104.481
            PM2.5       22.504   15.833  34.271     5.516   5.960   37.656
            SO2         12.200   10.061  11.723     7.195   6.332   18.319

In [31]:
latex = results_df.style.format("{:.2f}").to_latex(
    caption="Dataset statistics per network and pollutant (mean over sites).",
    label="tab:data_statistics",
    hrules=True,
)
with open("../output/data_statistics.tex", "w") as f:
    f.write(latex)
print("Saved to ../output/data_statistics.tex")


Saved to ../output/data_statistics.tex
